In [1]:
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torchinfo import summary

PyTorch's Dataset and DataLoader classes solve key training pipeline issues like memory inefficiency and lack of batching. The Notebooks explains transitioning from batch gradient descent to mini-batch gradient descent.

**Dataset Class Blueprint:**

```python
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self):      # Load data here (CSV/images/etc.)
    def __len__(self):       # Return # rows (e.g., 10)
    def __getitem__(idx):    # Return idx-th row (+transforms)
```

**DataLoader Workflow:**
```python
dataset = CustomDataset(X, y)      # Wraps data
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_features, batch_labels in dataloader:
    # ...
    # Train on batch (10 rows at once!)
```


**DataLoader Class:**

The DataLoader wraps a Dataset and handles batching, shuffling, and parallel loading for you.

**DataLoader Control Flow:**

At the start of each epoch, the DataLoader (if `shuffle=True`)
shuffles indices(using a sampler).
- It divides the indices into chunks of `batch_size`.
- For each index in the chunk, data samples are fetched from the Dataset object
- The samples are then collected and combined into a batch (using `collate_fn`)
- The batch is returned to the main training loop


### DataLoader Parameters

The DataLoader class in PyTorch comes with several parameters that allow you to customize
how data is loaded, batched, and preprocessed. Some of the most commonly used and
important parameters include:
- `dataset` (mandatory):
    -  The Dataset from which the DataLoader will pull data.
    - Must be a subclass of torch.utils.data.Dataset that implements `__getitem__` and `__len__`.

- `batch_size`:
    - How many samples per batch to load.
    - Default is 1.
    - Larger batch sizes can speed up training on GPUs but require more memory.


- `shuffle`:
    - If True, the DataLoader will shuffle the dataset indices each epoch.
    - Helpful to avoid the model becoming too dependent on the order of samples.

- `num_workers`:
    - The number of worker processes used to load data in parallel.
    - Setting num_workers > 0 can speed up data loading by leveraging multiple CPU
    - cores, especially if I/O or preprocessing is a bottleneck.

- `pin_memory`:
    - If True, the DataLoader will copy tensors into pinned (page-locked) memory before returning them.
    - This can improve GPU transfer speed and thus overall training throughput, particularly on CUDA systems.

- `drop_last`:
    - If True, the DataLoader will drop the last incomplete batch if the total number of samples is not divisible by the batch size.
    - Useful when exact batch sizes are required (for example, in some batch normalization scenarios).

- `collate_fn`:
    - A callable that processes a list of samples into a batch (the default simply stacks tensors).
    - Custom collate_fn can handle variable-length sequences, perform custom batching logic, or handle complex data structures.

- `sampler`:
    - sampler defines the strategy for drawing samples (e.g., for handling imbalanced classes, or custom sampling strategies).
    - batch_sampler works at the batch level, controlling how batches are formed.

Typically, you don’t need to specify these if you are using batch_size and shuffle.
However, they provide lower-level control if you have advanced requirements.



---
## Example

In [3]:
from sklearn.datasets import make_classification

In [8]:
# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=123    # For reproducibility
)

In [10]:
X.shape, X

((10, 2),
 array([[-0.12831709, -1.18968556],
        [-0.1522535 ,  2.20648133],
        [-0.7732662 ,  1.34414743],
        [ 0.35066574,  1.89223867],
        [ 1.42225579, -0.93564326],
        [-1.54309852, -0.40381336],
        [-1.04364535, -1.59420328],
        [-0.99234076,  0.08495255],
        [-0.52503756, -1.7132207 ],
        [ 0.79417498,  0.92684339]]))

In [11]:
y.shape, y

((10,), array([0, 1, 1, 1, 0, 0, 0, 1, 0, 1]))

In [12]:
# Convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [13]:
X,y

(tensor([[-0.1283, -1.1897],
         [-0.1523,  2.2065],
         [-0.7733,  1.3441],
         [ 0.3507,  1.8922],
         [ 1.4223, -0.9356],
         [-1.5431, -0.4038],
         [-1.0436, -1.5942],
         [-0.9923,  0.0850],
         [-0.5250, -1.7132],
         [ 0.7942,  0.9268]]),
 tensor([0, 1, 1, 1, 0, 0, 0, 1, 0, 1]))

In [14]:
from torch.utils.data import Dataset, DataLoader

## Dataset Class

In [46]:
class CustomDataset(Dataset): # inherit from Dataset Class

    def __init__(self, features, labels):

        self.features = features
        self.labels = labels

    def __len__(self):

        return self.features.shape[0]

    def __getitem__(self, index):

        return self.features[index], self.labels[index]
    

In [31]:
dataset = CustomDataset(X, y)

In [33]:
dataset, type(dataset)

(<__main__.CustomDataset at 0x1670fd8d0>, __main__.CustomDataset)

In [34]:
len(dataset), dataset[0]

(10, (tensor([-0.1283, -1.1897]), tensor(0)))

## DataLoader Class

In [44]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True) # shuffle the data

In [45]:
for batch_features, batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-"*30)

tensor([[-0.9923,  0.0850],
        [-0.7733,  1.3441]])
tensor([1, 1])
------------------------------
tensor([[ 0.7942,  0.9268],
        [-0.1283, -1.1897]])
tensor([1, 0])
------------------------------
tensor([[-0.1523,  2.2065],
        [-0.5250, -1.7132]])
tensor([1, 0])
------------------------------
tensor([[-1.5431, -0.4038],
        [ 1.4223, -0.9356]])
tensor([0, 0])
------------------------------
tensor([[ 0.3507,  1.8922],
        [-1.0436, -1.5942]])
tensor([1, 0])
------------------------------


---
### Real Example

In [47]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [71]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)


X_train_tensor = torch.from_numpy(X_train).to(torch.float32)
X_test_tensor = torch.from_numpy(X_test).to(torch.float32)
y_train_tensor = torch.from_numpy(y_train).to(torch.float32)
y_test_tensor = torch.from_numpy(y_test).to(torch.float32)

In [72]:
X_train_tensor.shape 

torch.Size([455, 30])

In [73]:
X_train_tensor.dtype

torch.float32

In [74]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [75]:
len(train_dataset), X_train_tensor.dtype, X_test_tensor.dtype

(455, torch.float32, torch.float32)

In [76]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [77]:
### NN in pytorch

class SimpleNN(nn.Module):

    def __init__(self, num_features): 

        super().__init__()

        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out
        

In [98]:
learning_rate = 0.1
epochs = 100

In [99]:
# create model
model = SimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

loss_function = nn.BCELoss()

In [100]:
# define loop
for epoch in range(epochs):
    
    for batch_features, batch_labels in train_loader: 
        # forward pass
        y_pred = model(batch_features)
    
        # loss calculate
        loss = loss_function(y_pred, batch_labels.view(-1,1))
    
        # clear gradients
        optimizer.zero_grad()
    
        # backward pass
        loss.backward()
    
        # parameters update
        optimizer.step()
    
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')
    # if epoch % 50 == 0 or epoch == epochs-1:
    #     print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.1959533393383026
Epoch: 2, Loss: 0.23394379019737244
Epoch: 3, Loss: 0.047546084970235825
Epoch: 4, Loss: 0.21414469182491302
Epoch: 5, Loss: 0.14153258502483368
Epoch: 6, Loss: 0.043006591498851776
Epoch: 7, Loss: 0.0456504262983799
Epoch: 8, Loss: 0.12291811406612396
Epoch: 9, Loss: 0.09386610239744186
Epoch: 10, Loss: 0.0977458506822586
Epoch: 11, Loss: 0.042821645736694336
Epoch: 12, Loss: 0.0273843202739954
Epoch: 13, Loss: 0.023664122447371483
Epoch: 14, Loss: 0.038227107375860214
Epoch: 15, Loss: 0.052627868950366974
Epoch: 16, Loss: 0.09183793514966965
Epoch: 17, Loss: 0.030574675649404526
Epoch: 18, Loss: 0.06263304501771927
Epoch: 19, Loss: 0.0442381426692009
Epoch: 20, Loss: 0.014810199849307537
Epoch: 21, Loss: 0.022546175867319107
Epoch: 22, Loss: 0.1924615204334259
Epoch: 23, Loss: 0.006102028768509626
Epoch: 24, Loss: 0.015440131537616253
Epoch: 25, Loss: 0.039812587201595306
Epoch: 26, Loss: 0.05390264466404915
Epoch: 27, Loss: 0.02240832708775997
Epoc

In [103]:
### Evaluation

In [105]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.6).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9688
